# Data Battle 2026 - Météorage
## Notebook 01 : Exploration des Données (EDA)

**Objectif** : Prédire la fin d'un orage (probabilité que l'éclair courant soit le dernier éclair nuage-sol d'une alerte)

### Colonnes du dataset
| Colonne | Description |
|---|---|
| `lightning_id` | Identifiant unique global de l'éclair |
| `lightning_airport_id` | Identifiant de l'éclair pour un aéroport donné |
| `date` | Horodatage UTC de l'éclair |
| `lon`, `lat` | Coordonnées géographiques de l'éclair |
| `amplitude` | Amplitude du courant en kA (négatif = polarité négative) |
| `maxis` | Courant max instantané (kA) |
| `icloud` | True si éclair intra-nuage, False si nuage-sol |
| `dist` | Distance à l'aéroport (km, max ~30km) |
| `azimuth` | Direction depuis l'aéroport (degrés) |
| `airport` | Nom de l'aéroport |
| `airport_alert_id` | ID de l'alerte (null si hors alerte) |
| `is_last_lightning_cloud_ground` | **CIBLE** : True si dernier éclair CG de l'alerte |


In [ ]:
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

TRAIN_PATH = 'segment_alerts_all_airports_train/segment_alerts_all_airports_train.csv'
EVAL_PATH  = 'segment_alerts_all_airports_eval.csv'
print('Librairies chargées ')

## 1. Chargement et aperçu général

In [ ]:
df = pd.read_csv(TRAIN_PATH)
df['date'] = pd.to_datetime(df['date'], utc=True)
df['icloud'] = df['icloud'].astype(bool)

print(f'Shape train : {df.shape}')
print(f'Période     : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Aéroports   : {df["airport"].unique()}')
df.head()

In [ ]:
print('=== STATISTIQUES GÉNÉRALES ===')
print(df.describe().round(3).to_string())
print('\n=== VALEURS MANQUANTES ===')
print(df.isnull().sum())
print('\n=== DISTRIBUTION CIBLE ===')
print(df['is_last_lightning_cloud_ground'].value_counts(dropna=False))

## 2. Analyse par aéroport

In [ ]:
# Stats globales par aéroport
airport_stats = df.groupby('airport').agg(
    nb_eclairs=('lightning_id', 'count'),
    nb_eclairs_CG=('icloud', lambda x: (~x).sum()),
    nb_eclairs_IC=('icloud', 'sum'),
    dist_moy=('dist', 'mean'),
    amplitude_moy=('amplitude', 'mean'),
    amplitude_abs_moy=('amplitude', lambda x: np.abs(x).mean()),
).reset_index()

# Ajouter alertes
alert_stats = df[df['airport_alert_id'].notnull()].groupby('airport').agg(
    nb_alertes=('airport_alert_id', 'nunique'),
    nb_eclairs_alertes=('lightning_id', 'count'),
).reset_index()

airport_stats = airport_stats.merge(alert_stats, on='airport', how='left')
airport_stats['pct_ratio_IC_CG'] = (airport_stats['nb_eclairs_IC'] / airport_stats['nb_eclairs_CG']).round(2)
print(airport_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
airports = df['airport'].unique()
colors = sns.color_palette('Set2', len(airports))

# Nombre d'éclairs par aéroport
ax = axes[0, 0]
counts = df['airport'].value_counts()
bars = ax.bar(counts.index, counts.values, color=colors)
ax.set_title('Nombre total d\'éclairs par aéroport')
ax.set_ylabel('Nombre d\'éclairs')
ax.tick_params(axis='x', rotation=20)

# Ratio intra-cloud vs cloud-ground
ax = axes[0, 1]
icloud_counts = df.groupby(['airport', 'icloud']).size().unstack(fill_value=0)
icloud_counts.columns = ['Cloud-Ground', 'Intra-Cloud']
icloud_counts.plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Répartition éclairs CG vs IC par aéroport')
ax.tick_params(axis='x', rotation=20)
ax.legend()

# Distribution de l'amplitude (abs) par aéroport
ax = axes[0, 2]
for airport, color in zip(airports, colors):
    data = df[df['airport'] == airport]['amplitude'].abs()
    data.hist(bins=50, alpha=0.5, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution amplitude |kA| par aéroport')
ax.set_xlabel('|Amplitude| (kA)')
ax.set_xlim(0, 100)
ax.legend(fontsize=8)

# Distribution de la distance par aéroport
ax = axes[1, 0]
for airport, color in zip(airports, colors):
    data = df[df['airport'] == airport]['dist']
    data.hist(bins=30, alpha=0.5, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution distance (km) par aéroport')
ax.set_xlabel('Distance (km)')
ax.legend(fontsize=8)

# Nombre d'alertes par aéroport
ax = axes[1, 1]
alert_counts = df[df['airport_alert_id'].notnull()].groupby('airport')['airport_alert_id'].nunique()
ax.bar(alert_counts.index, alert_counts.values, color=colors)
ax.set_title('Nombre d\'alertes par aéroport')
ax.set_ylabel('Nombre d\'alertes')
ax.tick_params(axis='x', rotation=20)

# Distribution maxis
ax = axes[1, 2]
for airport, color in zip(airports, colors):
    data = df[df['airport'] == airport]['maxis']
    data.hist(bins=40, alpha=0.5, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution maxis par aéroport')
ax.set_xlabel('Maxis (kA)')
ax.legend(fontsize=8)

plt.suptitle('Analyse par aéroport - Dataset d\'entraînement', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plots/01_airport_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Analyse temporelle

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

df['month'] = df['date'].dt.month
df['hour'] = df['date'].dt.hour
df['year'] = df['date'].dt.year
df['dayofweek'] = df['date'].dt.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Distribution mensuelle
ax = axes[0, 0]
monthly = df.groupby(['month', 'airport']).size().unstack(fill_value=0)
monthly.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Saisonnalité des éclairs (par mois)')
ax.set_xlabel('Mois')
ax.set_ylabel('Nombre d\'éclairs')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc'])

# Distribution horaire
ax = axes[0, 1]
hourly = df.groupby(['hour', 'airport']).size().unstack(fill_value=0)
hourly.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Distribution horaire des éclairs (UTC)')
ax.set_xlabel('Heure UTC')
ax.set_ylabel('Nombre d\'éclairs')

# Distribution annuelle
ax = axes[1, 0]
yearly = df.groupby(['year', 'airport']).size().unstack(fill_value=0)
yearly.plot(kind='bar', ax=ax, stacked=True)
ax.set_title('Distribution annuelle des éclairs')
ax.set_xlabel('Année')
ax.tick_params(axis='x', rotation=0)

# Distribution mois x heure (heatmap globale)
ax = axes[1, 1]
pivot = df.groupby(['month', 'hour']).size().unstack(fill_value=0)
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', fmt='d', cbar_kws={'label': 'Nb éclairs'})
ax.set_title('Heatmap : Mois × Heure')
ax.set_xlabel('Heure UTC')
ax.set_ylabel('Mois')
ax.set_yticklabels(['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc'], rotation=0)

plt.tight_layout()
plt.savefig('plots/02_temporal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analyse des alertes

In [ ]:
# Focus sur les données labelisées (dans des alertes)
df_alerts = df[df['airport_alert_id'].notnull()].copy()
df_alerts['is_last'] = df_alerts['is_last_lightning_cloud_ground'].map({'True': True, 'False': False})

# Stats par alerte
alert_analysis = df_alerts.groupby(['airport', 'airport_alert_id']).agg(
    nb_eclairs=('lightning_id', 'count'),
    nb_CG=('icloud', lambda x: (~x).sum()),
    nb_IC=('icloud', 'sum'),
    debut=('date', 'min'),
    fin=('date', 'max'),
    dist_min=('dist', 'min'),
    dist_max=('dist', 'max'),
    amplitude_max=('amplitude', lambda x: np.abs(x).max()),
).reset_index()

alert_analysis['duree_min'] = (alert_analysis['fin'] - alert_analysis['debut']).dt.total_seconds() / 60
print(f'Nb alertes : {len(alert_analysis)}')
print(f'\nDurée des alertes (minutes):')
print(alert_analysis['duree_min'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Durée des alertes
ax = axes[0, 0]
for airport, color in zip(airports, colors):
    data = alert_analysis[alert_analysis['airport'] == airport]['duree_min']
    data.hist(bins=30, alpha=0.6, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution durée des alertes (min)')
ax.set_xlabel('Durée (minutes)')
ax.set_xlim(0, 300)
ax.legend(fontsize=8)

# Nb éclairs par alerte
ax = axes[0, 1]
for airport, color in zip(airports, colors):
    data = alert_analysis[alert_analysis['airport'] == airport]['nb_eclairs']
    data.hist(bins=30, alpha=0.6, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution nb éclairs par alerte')
ax.set_xlabel('Nb éclairs')
ax.set_xlim(0, 300)
ax.legend(fontsize=8)

# Nb éclairs CG par alerte
ax = axes[0, 2]
for airport, color in zip(airports, colors):
    data = alert_analysis[alert_analysis['airport'] == airport]['nb_CG']
    data.hist(bins=30, alpha=0.6, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution nb éclairs CG par alerte')
ax.set_xlabel('Nb éclairs CG')
ax.legend(fontsize=8)

# Durée vs nb éclairs
ax = axes[1, 0]
for airport, color in zip(airports, colors):
    data = alert_analysis[alert_analysis['airport'] == airport]
    ax.scatter(data['nb_eclairs'], data['duree_min'], alpha=0.4, label=airport, color=color, s=20)
ax.set_title('Durée vs Nb éclairs par alerte')
ax.set_xlabel('Nb éclairs')
ax.set_ylabel('Durée (min)')
ax.set_xlim(0, 500)
ax.set_ylim(0, 400)
ax.legend(fontsize=8)

# Distribution saisonnière des alertes
ax = axes[1, 1]
alert_analysis['mois_debut'] = alert_analysis['debut'].dt.month
monthly_alerts = alert_analysis.groupby(['mois_debut', 'airport']).size().unstack(fill_value=0)
monthly_alerts.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Saisonnalité des alertes (par mois)')
ax.set_xlabel('Mois')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])

# Distance minimale pendant l'alerte
ax = axes[1, 2]
for airport, color in zip(airports, colors):
    data = alert_analysis[alert_analysis['airport'] == airport]['dist_min']
    data.hist(bins=30, alpha=0.6, ax=ax, label=airport, color=color, density=True)
ax.set_title('Distribution distance min (km) par alerte')
ax.set_xlabel('Distance minimale (km)')
ax.legend(fontsize=8)

plt.suptitle('Analyse des alertes', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plots/03_alert_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Analyse spatiale des éclairs

In [ ]:
# Coordonnées approx des aéroports (centre de masse des éclairs dans la zone de 5km)
airport_coords = {
    'Ajaccio':  (41.923, 8.802),
    'Bastia':   (42.552, 9.484),
    'Biarritz': (43.468, -1.530),
    'Nantes':   (47.154, -1.607),
    'Pise':     (43.683, 10.394),
}

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, (airport, color) in enumerate(zip(airports, colors)):
    ax = axes[i]
    data = df[df['airport'] == airport]
    ax.scatter(data['lon'], data['lat'], c='lightgray', s=0.3, alpha=0.3)
    
    # Éclairs en alerte
    data_alert = data[data['airport_alert_id'].notnull()]
    cg = data_alert[~data_alert['icloud']]
    ic = data_alert[data_alert['icloud']]
    
    ax.scatter(cg['lon'], cg['lat'], c='red', s=2, alpha=0.5, label='CG (alerte)')
    ax.scatter(ic['lon'], ic['lat'], c='orange', s=2, alpha=0.3, label='IC (alerte)')
    
    # Marqueur aéroport
    if airport in airport_coords:
        lat0, lon0 = airport_coords[airport]
        ax.scatter(lon0, lat0, marker='*', s=300, c='blue', zorder=5, label='Aéroport')
    
    # Cercle 30km approx
    theta = np.linspace(0, 2*np.pi, 100)
    if airport in airport_coords:
        lat0, lon0 = airport_coords[airport]
        km_per_deg_lat = 111
        km_per_deg_lon = 111 * np.cos(np.radians(lat0))
        circle_lons = lon0 + (30/km_per_deg_lon) * np.cos(theta)
        circle_lats = lat0 + (30/km_per_deg_lat) * np.sin(theta)
        ax.plot(circle_lons, circle_lats, 'b--', linewidth=1, alpha=0.5)
        # Cercle 3km
        circle_lons3 = lon0 + (3/km_per_deg_lon) * np.cos(theta)
        circle_lats3 = lat0 + (3/km_per_deg_lat) * np.sin(theta)
        ax.plot(circle_lons3, circle_lats3, 'g--', linewidth=1, alpha=0.5)
    
    ax.set_title(airport)
    ax.set_aspect('equal')
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Distribution spatiale des éclairs (bleu=30km, vert=3km danger)', fontsize=13)
plt.tight_layout()
plt.savefig('plots/04_spatial_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Analyse des patterns de fin d'alerte

In [ ]:
# Pour chaque alerte, analyser l'évolution temporelle des éclairs avant la fin
df_alerts_sorted = df_alerts.sort_values(['airport', 'airport_alert_id', 'date'])

# Calculer le temps depuis le dernier éclair précédent (inter-arrival time)
df_alerts_sorted['time_prev'] = df_alerts_sorted.groupby(
    ['airport', 'airport_alert_id'])['date'].diff().dt.total_seconds()

# Calculer le rang de l'éclair dans l'alerte (position relative)
df_alerts_sorted['rank_in_alert'] = df_alerts_sorted.groupby(
    ['airport', 'airport_alert_id']).cumcount()
df_alerts_sorted['total_in_alert'] = df_alerts_sorted.groupby(
    ['airport', 'airport_alert_id'])['lightning_id'].transform('count')
df_alerts_sorted['relative_pos'] = df_alerts_sorted['rank_in_alert'] / df_alerts_sorted['total_in_alert']

# Temps depuis le début de l'alerte
alert_start = df_alerts_sorted.groupby(['airport', 'airport_alert_id'])['date'].transform('min')
df_alerts_sorted['time_since_start'] = (df_alerts_sorted['date'] - alert_start).dt.total_seconds() / 60

print('Features temporelles calculées ')
print(df_alerts_sorted[['relative_pos', 'time_prev', 'time_since_start', 'is_last']].describe().round(2))

In [ ]:
# Comparaison : éclairs normaux vs dernier éclair
last_lightnings = df_alerts_sorted[df_alerts_sorted['is_last'] == True]
not_last = df_alerts_sorted[df_alerts_sorted['is_last'] == False]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Position relative dans l'alerte
ax = axes[0, 0]
ax.hist(not_last['relative_pos'], bins=30, alpha=0.6, label='Pas le dernier', density=True, color='steelblue')
ax.hist(last_lightnings['relative_pos'], bins=30, alpha=0.6, label='Dernier éclair', density=True, color='red')
ax.set_title('Position relative dans l\'alerte')
ax.set_xlabel('Position relative (0=début, 1=fin)')
ax.legend()

# Inter-arrival time
ax = axes[0, 1]
ax.hist(not_last['time_prev'].dropna().clip(0, 600), bins=50, alpha=0.6, label='Pas le dernier', density=True, color='steelblue')
ax.hist(last_lightnings['time_prev'].dropna().clip(0, 600), bins=50, alpha=0.6, label='Dernier éclair', density=True, color='red')
ax.set_title('Temps depuis éclair précédent (s)')
ax.set_xlabel('Secondes')
ax.legend()

# Distance au moment du dernier éclair
ax = axes[0, 2]
ax.hist(not_last['dist'], bins=30, alpha=0.6, label='Pas le dernier', density=True, color='steelblue')
ax.hist(last_lightnings['dist'], bins=30, alpha=0.6, label='Dernier éclair', density=True, color='red')
ax.set_title('Distance à l\'aéroport (km)')
ax.set_xlabel('Distance (km)')
ax.legend()

# Amplitude absolue
ax = axes[1, 0]
ax.hist(np.abs(not_last['amplitude']).clip(0, 100), bins=40, alpha=0.6, label='Pas le dernier', density=True, color='steelblue')
ax.hist(np.abs(last_lightnings['amplitude']).clip(0, 100), bins=40, alpha=0.6, label='Dernier éclair', density=True, color='red')
ax.set_title('Amplitude absolue (kA)')
ax.set_xlabel('|Amplitude| kA')
ax.legend()

# Maxis
ax = axes[1, 1]
ax.hist(not_last['maxis'].clip(0, 5), bins=40, alpha=0.6, label='Pas le dernier', density=True, color='steelblue')
ax.hist(last_lightnings['maxis'].clip(0, 5), bins=40, alpha=0.6, label='Dernier éclair', density=True, color='red')
ax.set_title('Maxis (kA)')
ax.set_xlabel('Maxis')
ax.legend()

# icloud distribution pour les derniers éclairs
ax = axes[1, 2]
x = ['Cloud-Ground', 'Intra-Cloud']
y_last = [last_lightnings[~last_lightnings['icloud']].shape[0], last_lightnings[last_lightnings['icloud']].shape[0]]
y_notlast = [not_last[~not_last['icloud']].shape[0], not_last[not_last['icloud']].shape[0]]
bars1 = ax.bar([0, 1], y_notlast, width=0.4, label='Pas le dernier', alpha=0.7, color='steelblue')
bars2 = ax.bar([0.4, 1.4], y_last, width=0.4, label='Dernier éclair', alpha=0.7, color='red')
ax.set_xticks([0.2, 1.2])
ax.set_xticklabels(['Cloud-Ground', 'Intra-Cloud'])
ax.set_title('Type d\'éclair : dernier vs autres')
ax.legend()

plt.suptitle('Comparaison : dernier éclair vs éclairs normaux', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plots/05_last_lightning_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Analyse d'une alerte individuelle (exemple)

In [ ]:
# Visualiser quelques alertes type
sample_alerts = df_alerts_sorted.groupby(['airport', 'airport_alert_id']).filter(
    lambda x: len(x) > 20 and len(x) < 100
)

# Prendre un exemple par aéroport
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, airport in enumerate(airports):
    ax = axes[i]
    airport_data = sample_alerts[sample_alerts['airport'] == airport]
    if len(airport_data) == 0:
        continue
    
    # Prendre la première alerte de taille raisonnable
    alert_id = airport_data['airport_alert_id'].iloc[0]
    alert_data = airport_data[airport_data['airport_alert_id'] == alert_id].copy()
    alert_data = alert_data.sort_values('date')
    
    t0 = alert_data['date'].min()
    times = (alert_data['date'] - t0).dt.total_seconds() / 60
    
    cg_mask = ~alert_data['icloud']
    ic_mask = alert_data['icloud']
    last_mask = alert_data['is_last'] == True
    
    ax.scatter(times[ic_mask], alert_data['dist'][ic_mask], c='orange', s=20, alpha=0.6, label='IC')
    ax.scatter(times[cg_mask], alert_data['dist'][cg_mask], c='red', s=30, alpha=0.8, label='CG')
    if last_mask.any():
        ax.scatter(times[last_mask], alert_data['dist'][last_mask], 
                   c='black', s=200, marker='*', zorder=5, label='Dernier CG')
    
    ax.axhline(y=3, color='green', linestyle='--', alpha=0.5, label='3km danger')
    ax.set_title(f'{airport}\nAlerte {int(alert_id)}')
    ax.set_xlabel('Temps (min)')
    ax.set_ylabel('Distance (km)')
    ax.legend(fontsize=7)

plt.suptitle('Exemples d\'alertes : Distance vs Temps (étoile = dernier éclair CG)', fontsize=13)
plt.tight_layout()
plt.savefig('plots/06_alert_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Corrélations et matrice de features

In [ ]:
# Préparation d'une matrice de features pour analyse de corrélation
feature_cols = ['dist', 'amplitude', 'maxis', 'relative_pos', 'time_since_start']

df_feat = df_alerts_sorted[feature_cols + ['is_last', 'icloud', 'airport']].copy()
df_feat['is_last_int'] = df_feat['is_last'].astype(int)
df_feat['icloud_int'] = df_feat['icloud'].astype(int)
df_feat['amp_abs'] = np.abs(df_feat['amplitude'])

corr_matrix = df_feat[['dist', 'amp_abs', 'maxis', 'relative_pos', 'time_since_start', 'icloud_int', 'is_last_int']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, mask=mask,
            xticklabels=['dist', '|amp|', 'maxis', 'rel_pos', 'time_start', 'icloud', 'is_last'],
            yticklabels=['dist', '|amp|', 'maxis', 'rel_pos', 'time_start', 'icloud', 'is_last'])
ax.set_title('Matrice de corrélation des features')
plt.tight_layout()
plt.savefig('plots/07_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Résumé et hypothèses

In [ ]:
print('=== RÉSUMÉ EDA - DATA BATTLE 2026 ===')
print(f'''
DATASET:
  - Train: {len(df):,} éclairs (2016-2022), 5 aéroports
  - Eval:  ~188k éclairs (2023-2025)
  - Alertes labellisées (train): {df['airport_alert_id'].notnull().sum():,} éclairs dans {df['airport_alert_id'].nunique():.0f} alertes

DÉSÉQUILIBRE:
  - Derniers éclairs CG (True):  {(df["is_last_lightning_cloud_ground"]=="True").sum():,}
  - Éclairs non-derniers (False): {(df["is_last_lightning_cloud_ground"]=="False").sum():,}
  - Ratio: ~{(df["is_last_lightning_cloud_ground"]=="True").sum() / (df["is_last_lightning_cloud_ground"]=="False").sum() * 100:.1f}% de positifs

FEATURES POTENTIELLES IDENTIFIÉES:
  1. Position relative dans l'alerte (rank/total)
  2. Temps depuis le début de l'alerte
  3. Temps inter-éclairs (temps depuis l'éclair précédent)
  4. Distance à l'aéroport
  5. Amplitude absolue (intensité)
  6. Maxis
  7. Type (icloud vs CG)
  8. Statistiques glissantes sur fenêtre temporelle (nb éclairs/heure, dist moyenne, etc.)
  9. Tendance : l'orage s'approche ou s'éloigne ?
  10. Saisonnalité (mois, heure)

HYPOTHÈSES:
  - Les derniers éclairs tendent à être plus espacés temporellement
  - La distance tend à augmenter vers la fin (orage qui s'éloigne)
  - Le ratio IC/CG peut indiquer l'intensité résiduelle
  - Chaque aéroport a des patterns saisonniers différents

BASELINE MÉTÉORAGE: 80.77% de précision avec méthodes statistiques pures
''')

In [ ]:
# Vérification finale : eval dataset
df_eval = pd.read_csv(EVAL_PATH)
df_eval['date'] = pd.to_datetime(df_eval['date'], utc=True)
print('=== EVAL DATASET ===')
print(f'Shape: {df_eval.shape}')
print(f'Période: {df_eval["date"].min().date()} → {df_eval["date"].max().date()}')
print(f'Alertes: {df_eval["airport_alert_id"].notnull().sum():,} éclairs dans {df_eval["airport_alert_id"].nunique():.0f} alertes')
print(f'Cible: {df_eval["is_last_lightning_cloud_ground"].value_counts(dropna=False).to_dict()}')